![](img/logo_ucm.jpg)

# Model Serving

El componente **Model Serving** de MLflow permite desplegar un modelo registrado directamente como un servicio web. Es decir, puedes exponer tu modelo como una API REST para hacer predicciones en tiempo real, de forma rápida y reproducible y sin desarrollar nosotros la API.

MLflow Model Serving:

    - Carga un modelo registrado en el Model Registry

    - Levanta un servidor HTTP que expone el modelo como endpoint REST

    - Permite hacer inferencias síncronas mediante peticiones HTTP (_/invocations_)

    - Opcionalmente valida el tipo de entrada mediante la firma del modelo (input/output schema)


Os recordará a lo que vimos anteriormente con FastAPI.

## Paso 1: Recuperar el modelo previamente registrado
Asegúrate de tener un modelo registrado en MLflow, de lo contrario obtendréis la siguiente notificación: 

> No hay modelos registrados en este registry URI.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# Ajustándolo al backend que estamos usando realmente
mlflow.set_tracking_uri("./mlruns")
mlflow.set_registry_uri("./mlruns")

client = MlflowClient()

registered_models = client.search_registered_models(max_results=100)

if not registered_models:
    print("No hay modelos registrados en este registry URI.")
else:
    for model in registered_models:
        print(f"Modelo: {model.name}")
        versions = client.search_model_versions(f"name='{model.name}'")
        for v in versions:
            uri = f"models:/{model.name}/{v.version}"
            stage = getattr(v, "current_stage", "N/A")  # compatibilidad
            print(f" - Versión: {v.version}")
            print(f"   Stage: {stage}")
            print(f"   Status: {v.status}")
            print(f"   URI: {uri}")


Modelo: mynewmodel
 - Versión: 1
   Stage: None
   Status: READY
   URI: models:/mynewmodel/1


In [7]:
registered_models[0].name

'mynewmodel'

Podemos añadir aliases al modelo, prueba a darle el alias de `@champion`

In [9]:
model_name = registered_models[0].name

# Get all versions of the model
models = client.search_model_versions(f"name='{model_name}'")

champions = []

for model in models:
    uri = f"models:/{model.name}/{model.version}"
    print(f"Model: {model.name} | Version: {model.version} | Alias: {model.aliases} | Stage: {model.current_stage} | URI: {uri}")
    if model.aliases == ['champion']:
        champions.append(model)


Model: mynewmodel | Version: 2 | Alias: ['champion'] | Stage: None | URI: models:/mynewmodel/2
Model: mynewmodel | Version: 1 | Alias: [] | Stage: None | URI: models:/mynewmodel/1


## Paso 2: Servir el modelo con MLflow
Usamos el siguiente comando en terminal para servir el modelo en el puerto 1234:

```bash
mlflow models serve -m runs:/<RUN_ID>/model -p 1234 --no-conda
```

Reemplaza `<RUN_ID>` por el ID impreso en la celda anterior. 

> Como ya nos ocurrió en anteriores Notebooks, no queremos bloquear el hilo principal así que utilizaremos un **subproces**:

In [11]:
champions[-1]

<ModelVersion: aliases=['champion'], creation_timestamp=1778931073043, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1778931073043, metrics=None, model_id='', name='mynewmodel', params=None, run_id='9efd7a9672c04672aaab25c57174abb9', run_link='', source='models:/mynewmodel/1', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [12]:
import subprocess

run_id = champions[-1].run_id 
port = 1234

# Comando para servir el modelo
command = [
    "mlflow", "models", "serve",
    "-m", f"runs:/{run_id}/model",
    "-p", str(port),
    "--no-conda"
]

# Ejecutar en segundo plano sin bloquear el notebook
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print(f"MLflow model serving launched on port {port} (PID={process.pid})")


MLflow model serving launched on port 1234 (PID=142708)


Implicaciones del uso de `--no-conda`:

MLflow usa el entorno actual (donde estás ejecutando el comando) para cargar y servir el model obviando el fichero conda.yaml o requirements.txt que vimos en etapas previas. Si no están, fallará en runtime (import errors, incompatibilidades, etc.).

Si no incluimos esta propiedad, MLflow creará un entorno por nosotros, lo que puede llevar un tiempo y requerirá tener `conda` instalado y con permisos para crear entornos e instalar dependencias automáticamente. MLflow puede usar además `virtualenv` o `pip` directamente. En definitiva, `--no-conda` es más rápido y menos reproducible. 

## Paso 3: Hacer una predicción vía API REST:

In [15]:
import requests
import json
from sklearn.datasets import load_iris

# El tráfico va a localhost por http sin cifrar. 
url = "http://127.0.0.1:1234/invocations"
headers = {"Content-Type": "application/json"}

# El anterior formato usado generará un error:
# data = {
#     'sepal_length': 6.9,
#     'sepal_width': 3.1,
#     'petal_length': 5.4,
#     'petal_width': 2.1,   
# }
# Error if we are not using the correct format: {'error_code': 'BAD_REQUEST', 'message': "The input must be a JSON dictionary with exactly one of the input fields {'inputs', 'dataframe_records', 'instances', 'dataframe_split'}. Received dictionary with input fields: []. IMPORTANT: The MLflow Model scoring protocol has changed in MLflow version 2.0. If you are seeing this error, you are likely using an outdated scoring request format. To resolve the error, either update your request format or adjust your MLflow Model's requirements file to specify an older version of MLflow (for example, change the 'mlflow' requirement specifier to 'mlflow==1.30.0'). If you are making a request using the MLflow client (e.g. via `mlflow.pyfunc.spark_udf()`), upgrade your MLflow client to a version >= 2.0 in order to use the new request format. For more information about the updated MLflow Model scoring protocol in MLflow 2.0, see https://mlflow.org/docs/latest/models.html#deploy-mlflow-models."}

# El formato actual no es tan estándar pero permite enviar datos con estructura similar al que tendríamos con un Dataframe. 
data = {
    "dataframe_split": {
        "columns": ["sepal_length", "sepal_width", "petal_length", "petal_width"],
        "data": [[6.9, 3.1, 5.4, 2.1]]
    }
}

response = requests.post(url=url, headers=headers, json=data)
print(f'Encoded values: {response.json()}')

target_names = load_iris().target_names
print(target_names)

Encoded values: {'predictions': [[0.0008213153923861682, 0.0020000331569463015, 0.9971786737442017]]}
['setosa' 'versicolor' 'virginica']


In [18]:
import numpy as np

pred = response.json()["predictions"][0]
print("pred:", pred, "type:", type(pred))

# En caso de que venga likelihood por clase -> argmax
if isinstance(pred, (list, tuple, np.ndarray)):
    class_idx = int(np.argmax(pred))
else:
    class_idx = int(pred)

predicted_label = target_names[class_idx]
print(f"Decoded values: {predicted_label}")


pred: [0.0008213153923861682, 0.0020000331569463015, 0.9971786737442017] type: <class 'list'>
Decoded values: virginica


## Paso 4: ¿Y si queremos desplegar múltiples modelos?

Puedes lanzar varias instancias de mlflow models serve, una por cada modelo, en puertos diferentes.

```bash
mlflow models serve -m "models:/modelo_1/Production" -p 1234 --no-conda &
mlflow models serve -m "models:/modelo_2/Production" -p 1235 --no-conda 
```

Es sencillo y rápido y se trata cada modelo de manera independiente. Sin embargo, necesitaremos controlar los puertos y procesos manualmente (tedioso), la escalabilidad está limitada al número de puertos disponibles si usas muchos modelos.

### Posibles soluciones 

Utilizar microservicions en FastAPI como vimos en las anteriores secciones, reutilizando un único puerto, aumentando el ancho de banda, escalabilidad: 

>

```python
from fastapi import FastAPI
import mlflow.pyfunc
import pandas as pd

app = FastAPI()


model_cache = {}

# Cargamos los modelos en memoria cuando arrancamos la APP. Asumimos que no van a venir nuevos modelos sin despliegue previo. 
@app.on_event("startup")
def load_models():
    global model_cache
    model_cache["iris_model"] = mlflow.pyfunc.load_model("models:/iris_model/Production")
    model_cache["otro_modelo"] = mlflow.pyfunc.load_model("models:/otro_modelo/Production")

class PredictionRequest(BaseModel):
    features: dict

# Pudiendo añadir el alias del modelo como parámetro de entrada también o dejarlo fijado como en este caso.
@app.post("/predict/{model_name}")
def predict(model_name: str, request: PredictionRequest):
    model = model_cache.get(model_name)
    if model is None:
        return {"error": f"Modelo '{model_name}' no encontrado"}
    df = pd.DataFrame([request.features])
    prediction = model.predict(df)
    return {"prediction": prediction.tolist()}
    
```
>
Para la gestión de múltiples requests simultáneas, deberemos levantar la app con suficientes recursos: 


```bash
uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4
```
>

O si queremos que dependa del número de núcleos del sistema: 

> 

```python

import multiprocessing
import uvicorn

if __name__ == "__main__":
    num_workers = multiprocessing.cpu_count()
    uvicorn.run("app:app", host="0.0.0.0", port=8000, workers=num_workers)

```

`````{admonition} Regla de escalado
:class: tip
* Para modelos ligeros o I/O-bound: usar tantos workers como núcleos (CPU-bound = más escalado).
* Para modelos grandes o con alto uso de RAM: usar menos workers para evitar saturar la memoria.
Por ejemplo: 
> max(2, int(multiprocessing.cpu_count() * 0.75))
`````

Dependerá de las pruebas de rendimiento que hagamos en los entornos de preproducción desde donde podremos tener una foto clara de los recursos que necesitaremos.

### Prueba de rendimiento simple: 


```bash
ab -n 100 -c 10 http://localhost:8000/predict/iris_model
```

**Apache Bench (ab)** es una herramienta muy simple y útil para hacer pruebas de carga (**benchmarking**) sobre un servicio de predicción en FastAPI.

>
Este comando lanza:

- 100 peticiones HTTP en total (-n 100)

- con un nivel de concurrencia de 10 peticiones al mismo tiempo (-c 10)

Lo que nos permitirá medir estadísticos como: Requests per second, Time per request o Percentiles de latencia (útil para SLA: 95%, 99%).

# MLflow en entornos cloud

`````{admonition} MLflow en cloud
:class: info
Hasta ahora, hemos visto cómo registrar modelos de forma local y cómo cargarlos y desplegarlos con **MLflow serve**. Sin embargo, cuando trabajamos en entornos colaborativos o en producción, es fundamental utilizar el **Model Registry** remoto (no en local) para gestionar versiones de modelos entre los integrantes del equipo, hacer seguimiento de su ciclo de vida y facilitar su transición entre etapas como `"Staging"` y `"Production"`. En la siguiente sección vamos a revisar qué diferencias podemos encontrarnos entre **MLflow Open Source** y el **MLflow Managed** que podemos encontrar en la nube (por ejemplo, en **Databricks**).
`````


`````{admonition} Comparartiva entre MLflow OSS vs Cloud
:class: note

| Característica                     | MLflow Open Source                        | MLflow en Databricks (Managed)                       |
|-----------------------------------|-------------------------------------------|-----------------------------------------------------|
| **Tracking Server**               | Autohospedado, requiere configuración     | Gestionado por Databricks                          |
| **UI Web**                        | Local o remota, simple                    | Integrada con Workspaces, vista enriquecida        |
| **Registry**                      | Básico, sin control de acceso             | Soporte completo con ACLs, comentarios, historial   |
| **Versionado de Modelos**        | Manual con `MlflowClient`                | Integrado, control de versiones automático         |
| **Etapas del modelo**            | Soporte básico (`Staging`, `Production`)  | Transición con validaciones, CI/CD integraciones   |
| **Notificaciones / Webhooks**    | No disponible directamente                | Disponible para automatizar flujos (CI, alerts)    |
| **Seguridad / Usuarios**         | Gestión manual                            | Control de acceso granular (RBAC)                  |
| **Almacenamiento de artefactos** | Local / S3 / GCS configurado manualmente  | Integrado con DBFS (Databricks File System)        |
| **Despliegue**                   | Requiere servir manualmente               | Integración con endpoints REST, real-time serving  |
| **Auditoría**                    | Limitada                                  | Completa (acciones, usuarios, timestamps)          |

> Si estás trabajando en equipos pequeños o académicos, MLflow OSS puede ser suficiente. Pero para despliegues empresariales, el Managed MLflow de Databricks ofrece una solución robusta y lista para producción.

`````

## Reflexión: Gobernanza, seguridad y escalabilidad en proyectos de Machine Learning

Uno de los aspectos más subestimados en los proyectos de Machine Learning es la **gobernanza y seguridad de los modelos**. A medida que los modelos pasan del entorno de desarrollo local al despliegue en producción, surgen necesidades críticas que van más allá de la experimentación y el ajuste de hiperparámetros.

Cuando trabajamos en entornos colaborativos o empresariales, no es suficiente con que un modelo funcione: es esencial saber **quién lo entrenó, con qué datos, en qué entorno, y con qué versiones**. Esta trazabilidad no solo es clave para la reproducibilidad, sino también para la **auditoría, cumplimiento normativo y control de calidad**.

### Gestión de accesos y roles

Contar con una **gestión de roles basada en permisos (RBAC)** permite establecer un control preciso sobre:

- Quién puede registrar modelos nuevos.
- Quién puede promover versiones a producción.
- Quién puede hacer rollback o eliminar modelos.

Este control granular **evita errores operativos**, garantiza que solo usuarios autorizados modifiquen artefactos críticos y es una piedra angular para ambientes regulados como banca, sanidad o industria farmacéutica. De utilizar versiones OSS, tendríamos que implementar estas políticas desde la propia compañía, en muchos casos "reinventando la rueda" puesto que ya existen herramientas que se integran perfectamente.

### Integración de datos, modelos y despliegue

En muchas organizaciones, los datos viven en un sistema, los modelos en otro, y el despliegue ocurre con scripts manuales. Esta fragmentación **limita la escalabilidad** y **aumenta el riesgo de inconsistencias**.

Trabajar en cloud permite tener todo integrado: desde el tracking de experimentos, el versionado de datasets, hasta el registro y despliegue del modelo. Esto no solo mejora la eficiencia, sino que reduce la complejidad operativa y favorece la colaboración entre equipos. Además, hasta ahora, hemos utilizado datasets de ejemplo, si estuvieramos descargando datos a un entorno local para realizar los experimentos, **estaríamos perdiendo la trazabilidad de los datos**.

### Exposición segura de modelos en la nube

Tener la capacidad de exponer modelos como APIs en la nube trae enormes ventajas: acceso desde cualquier punto del sistema, escalado dinámico, integración con pipelines de negocio, etc. Pero también plantea desafíos de seguridad:

- ¿Quién puede invocar el modelo?
- ¿Qué pasa si se expone información sensible?
- ¿Cómo protegemos la infraestructura ante un uso malicioso?

Por eso es importante que el **serving de modelos** esté protegido con autenticación, monitorizado, y gestionado dentro de un entorno controlado.


### Conclusión General

El verdadero reto en Machine Learning no es solo construir buenos modelos, sino **operarlos de forma segura, trazable y escalable**. La capacidad de gestionar accesos, roles, versiones y despliegue de modelos no es un lujo, sino una necesidad para quienes quieren llevar el ML más allá del notebook, hacia entornos reales donde los modelos tienen impacto.
